In [3]:
#from Processor import get_dataset#, get_temprel_counts
from Reader import TBDenseReader, TempEval3Reader, MAVENReader, OzRockReader, TweetsReader, WikiWarsReader
from copy import deepcopy
def get_temprel_props(data, type="by_set"):
    total = {}
    if type == "by_set":
        for split in data:
            counts = {}
            for sample in data[split]:
                for temprel in sample["ee_temprels"]:
                    if temprel["rel"] in ["BEFORE", "AFTER", "CONTAINS"]:
                        counts["BIG"] = counts.get("BIG", 0) + 1
                    else:
                        counts["SMALL"] = counts.get("SMALL", 0) + 1
        total[split] = counts
    else:
        for split in data:
            for sample in data[split]:
                for temprel in sample["ee_temprels"]:
                    if temprel["rel"] in ["BEFORE", "AFTER", "CONTAINS"]:
                        total["BIG"] = total.get("BIG", 0) + 1
                    else:
                        total["SMALL"] = total.get("SMALL", 0) + 1
    return total

def get_sample_temprel_ratio_class(sample):
    total = {"BIG": 0, "SMALL": 0}
    for temprel in sample["ee_temprels"]:
        if temprel["rel"] in ["BEFORE", "AFTER", "CONTAINS"]:
            total["BIG"] += 1
        else:
            total["SMALL"] += 1
    try:
        ratio = total["SMALL"]/(total["SMALL"]+total["BIG"])
    except ZeroDivisionError:
        ratio = 0
        print(sample)
        raise ZeroDivisionError("No temporal relations in sample")
    # Need to find ratio thresholds -> precompute it with whole dataset
    if ratio < 0.2: return "TINY"
    elif ratio < 0.4: return "SMALL"
    elif ratio < 0.6: return "MEDIUM"
    elif ratio < 0.8: return "BIG"
    else: return "HUGE"

def data_temprel_select(data):
    counts = {"BEFORE": 0, "AFTER": 0, "DURING": 0, "CONTAINS": 0, "OVERLAPS": 0, "EQUALS": 0, "IDENTITY": 0}
    target = {"BEFORE": 75855, "AFTER": 75855, "DURING": 75855, "CONTAINS": 75855, "OVERLAPS": 75855, "EQUALS": 75855, "IDENTITY": 75855}
    balanced = {}

    for name in data:
        balanced[name] = {}
        for split in data[name]:
            balanced[name][split] = []
            for sample in data[name][split]:
                new_sample = sample.copy()
                new_sample["ee_temprels"] = []
                for temprel in sample["ee_temprels"]:

                    if name in ["TempEval3", "TBDense"] and split == "train" and temprel['rel'] == "BEFORE":
                        continue

                    elif temprel['rel'] == "BEFORE" and counts["BEFORE"] == target["BEFORE"] and counts["AFTER"] < target["AFTER"]:
                        counts["AFTER"] += 1
                        new_sample["ee_temprels"].append({"rel": "AFTER", "e1": temprel["e2"], "e2": temprel["e1"]})

                    elif counts[temprel['rel']] < target[temprel['rel']]:
                            counts[temprel['rel']] += 1
                            new_sample["ee_temprels"].append(temprel)

                balanced[name][split].append(new_sample)

    return balanced, counts

In [ ]:
def reindex(data):
    eid2index = {}
    tid2index = {}
    for inst in data["instances"]:
        if inst["type"] == "EVENT":
            eid2index[inst["id"]] = len(eid2index)
            inst['id'] = eid2index[inst['id']]
        else:
            tid2index[inst["id"]] = len(tid2index)
            inst['id'] = tid2index[inst['id']]

    for temprel in data["ee_temprels"]:
        temprel['e1'] = eid2index[temprel['e1']]
        temprel['e2'] = eid2index[temprel['e2']]

    for eventtimes in data["event_times"]:
        if eventtimes['time'][0] == "e":
            time = eventtimes['event']
            event = eventtimes['time']
            eventtimes['event'] = event
            eventtimes['time'] = time

        eventtimes['event'] = eid2index[eventtimes['event']]
        eventtimes['time'] = tid2index[eventtimes['time']]

    return data

def get_dataset(reader):
    data = reader.read()
    if type(reader) is MAVENReader:
        data.pop("test")
    for split in data:
        for sample in data[split]:
            sample = reindex(sample)
    return data

def obtain_combined_data(data):
    data = {}
    rawdata_path = "D:\\GeoTKG\\rawdata\\"
    for name, reader in [("TempEval3", TempEval3Reader),("TBDense", TBDenseReader),('MAVEN_ERE', MAVENReader),]:
        data[name] = get_dataset(reader(rawdata_path + name))
    dc = deepcopy(data)
    bal, cnts = data_temprel_select(dc)
    

In [ ]:
# Do time norm datasets and stratify by type
# Add NONE event time relationships

count = 0
found_evs = []
for dset in dc:
    for split in dc[dset]:
        for sample in dc[dset][split]:
            sample_evs = []
            for et in sample["event_times"]:
                count += 1
                sample_evs.append(et['event'])
            found_evs.append(sample_evs)



0
9
0
3
4
6
7
8
9
12
14
17
18
17
5
11
0
2
0
2
3
5
0
1
17
20
26
1
15
57
61
81
92
95
98
101
2
8
20
22
23
33
86
88
89
100
1
2
5
7
15
14
9
19
0
4
6
29
35
12
27
3
2
10
13
14
19
22
24
28
30
33
36
37
38
8
0
12
13
14
19
17
20
21
26
27
31
9
3
0
4
4
5
7
8
9
10
12
14
15
17
0
5
6
7
9
10
11
12
14
16
18
19
20
21
22
23
24
26
31
32
37
38
41
45
49
52
53
54
55
56
57
60
61
62
63
65
68
70
71
73
1
4
5
7
8
9
10
11
12
13
16
17
18
19
20
25
26
28
30
31
32
33
35
36
38
0
1
2
4
7
8
9
1
2
0
0
3
4
5
6
7
8
9
10
11
16
17
18
19
20
21
22
23
24
25
26
27
29
31
31
31
31
32
33
35
36
37
39
40
43
44
45
46
48
50
52
53
55
60
61
63
64
65
67
68
69
70
74
75
78
79
2
4
5
12
15
16
18
20
21
22
24
26
27
31
33
34
36
0
1
2
5
6
7
8
9
11
13
14
15
17
21
22
23
24
25
26
27
28
29
30
32
35
38
39
49
51
52
53
55
56
57
60
61
69
70
72
73
74
75
76
77
78
0
1
2
3
4
6
7
8
9
11
12
13
17
18
19
20
23
25
26
28
29
31
32
39
40
41
43
45
46
48
48
52
53
54
55
49
50
56
57
58
59
0
1
2
3
9
10
11
12
13
16
17
18
22
23
25
26
27
31
32
33
34
35
36
38
40
41
42
43
44
45

98330

In [23]:
for dset in dc:
    for split in dc[dset]:
        for sample, thing in zip(dc[dset][split], found_evs):
            if len(sample["event_times"]) != len(thing):
                print(len(sample["event_times"]), len(thing))

10 2
11 14
5 2
3 0
9 4
4 5
2 19
19 8
3 7
6 15
5 12
16 13
17 40
4 25
18 7
7 59
10 17
13 45
10 41
9 65
3 2
6 14
18 2
1 0
11 4
14 5
16 19
5 8
3 7
14 15
10 12
10 13
7 40
5 25
15 7
1 59
9 17
4 45
4 41
7 65
11 10
2 9
13 52
8 21
1 15
8 16
7 32
7 20
2 15
7 27
16 34
4 32
4 20
4 25
18 5
8 11
2 24
15 10
1 37
8 11
1 60
4 52
11 116
8 28
8 50
13 165
19 65
8 53
7 13
9 8
4 11
2 20
7 5
12 21
11 28
6 12
3 28
15 6
4 26
11 9
5 4
2 7
3 4
18 5
3 10
9 7
1 18
18 24
1 26
2 14
4 24
19 31
8 51
5 12
4 3
13 2
17 6
4 40
3 26
3 12
11 7
7 6
13 25
14 2
1 3
2 7
2 1
11 12
23 22
11 18
1 4
5 25
3 2
3 7
10 2
7 16
4 5
2 3
8 13
1 8
7 6
3 10
10 25
3 8
19 4
2 9
10 6
19 7
4 1
8 4
14 8
5 1
15 6
5 8
8 9
5 3
9 8
8 10
3 6
14 2
7 26
3 38
6 0
5 3
14 7
4 6
6 4
12 1
11 3
1 8
1 10
20 6
2 3
15 9
9 6
8 5
1 2
4 2
9 3
7 5
11 2
6 4
6 8
5 3
16 4
5 4
10 4
22 3
2 6
15 4
18 9
18 5
7 3
9 4
2 1
2 1
6 8
17 4
3 4
1 2
2 4
8 6
12 1
10 4
12 3
1 6
9 7
8 4
9 8
3 6
5 7
17 3
22 1
1 8
6 8
10 25
13 14
2 21
9 16
3 21
13 34
6 22
1 30
7 13
7 3
6 1
8 2
2 3
7 1
9

In [11]:
tb_cnt = get_temprel_props(bal['TBDense'], type="whole")
te_cnt = get_temprel_props(bal['TempEval3'], type="whole")
m_cnt = get_temprel_props(bal['MAVEN_ERE'], type="whole")

print(m_cnt["BIG"]+m_cnt["SMALL"])

237547


In [ ]:
for dset in dc:
    for splt in dc[dset]:
        print(f'{dset} - {splt}: {len(bal[dset][splt]) == len(dc[dset][splt])}')

In [ ]:
# import json
# for name, set in [("train.json", X_train), ("eval.json", X_val), ("test.json", X_test)]:
#     with open("D:\\GeoTKG\\cleandata\\"+name, 'w') as json_file:
#         json.dump(set, json_file, indent=4)
# with open("C:\\Users\\hazza\\OneDrive\\Desktop\\GeoTKG\\cleandata\\data\\"+"name.jsonl", 'w') as json_file:
#     for line in tester:
#         json_file.write(json.dumps(line)+"\n")

In [ ]:
from sklearn.model_selection import train_test_split
all_data = []
y = []
for name in bal:
    for split in bal[name]:
        for sample in bal[name][split]:
            print(name, split)
            all_data.append(sample)
            y.append(get_sample_temprel_ratio_class(sample))

X_train, X_test, y_train, y_test = train_test_split(all_data, y, test_size=0.1, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, stratify=y_train)